# NB10B — Topic Sentiment Aggregation
## Topic-Sentiment Extension | MSc Dissertation
### Pavlos Papachristos | University of Essex

**Purpose:** Aggregate topic-specific FinBERT sentiment by country-year. 
Merge with JST macro panel to produce the feature matrix for NB10C.

In [1]:
import pandas as pd
import numpy as np
import os
import re

DATA_DIR  = r"C:\Users\Owner\OneDrive\dissertation\data\processed"
TOPIC_DIR = r"C:\Users\Owner\OneDrive\dissertation\data\processed\topic_sentiment"
OUT_DIR   = TOPIC_DIR

TOPICS = ["credit", "real_estate", "monetary_policy", "sovereign", "corporate"]

# Load full tagged file
bis = pd.read_csv(os.path.join(TOPIC_DIR, "bis_topic_tagged.csv"))
print(bis.shape)
print(bis.columns.tolist())

(16622, 10)
['url', 'year', 'date', 'author', 'description', 'P_pos', 'P_neg', 'P_neutral', 'topics', 'n_topics']


In [2]:
# ── Central bank name → ISO3 mapping ─────────────────────────────────────────
CB_MAP = {
    "federal reserve": "USA", "bank of england": "GBR",
    "european central bank": "DEU", "bundesbank": "DEU",
    "bank of france": "FRA", "banque de france": "FRA",
    "bank of italy": "ITA", "banca d'italia": "ITA",
    "bank of japan": "JPN", "bank of canada": "CAN",
    "reserve bank of australia": "AUS",
    "reserve bank of new zealand": "NZL",
    "swiss national bank": "CHE",
    "riksbank": "SWE", "bank of sweden": "SWE",
    "norges bank": "NOR", "bank of norway": "NOR",
    "danmarks nationalbank": "DNK", "bank of denmark": "DNK",
    "bank of finland": "FIN", "suomen pankki": "FIN",
    "bank of netherlands": "NLD", "nederlandsche bank": "NLD",
    "national bank of belgium": "BEL", "banque nationale de belgique": "BEL",
    "bank of portugal": "PRT", "banco de portugal": "PRT",
    "bank of spain": "ESP", "banco de españa": "ESP",
    "bank of greece": "GRC",
    "irish": "IRL",
    "reserve bank of india": "IND",
    "bank of korea": "KOR",
    "monetary authority of singapore": "SGP",
    "hong kong monetary": "HKG",
    "peoples bank of china": "CHN", "people's bank of china": "CHN",
    "bank of mexico": "MEX", "banco de mexico": "MEX",
    "central bank of brazil": "BRA", "banco central do brasil": "BRA",
}

def extract_iso(description):
    if not isinstance(description, str):
        return None
    desc_lower = description.lower()
    for cb, iso in CB_MAP.items():
        if cb in desc_lower:
            return iso
    return None

bis["iso"] = bis["description"].apply(extract_iso)
bis["year"] = pd.to_numeric(bis["year"], errors="coerce").astype("Int64")

print("ISO coverage:", bis["iso"].notna().sum(), "of", len(bis))
print("\nTop countries:")
print(bis["iso"].value_counts().head(15))

ISO coverage: 11535 of 16622

Top countries:
iso
DEU    2947
USA    2089
IND     782
JPN     663
GBR     655
CAN     502
AUS     481
SWE     470
CHE     367
FRA     321
ITA     294
ESP     274
NOR     262
SGP     245
HKG     230
Name: count, dtype: int64


In [3]:
# ── Aggregate topic sentiment by country-year ─────────────────────────────────
# Keep only speeches with ISO and sentiment scores
bis_scored = bis.dropna(subset=["iso", "P_pos", "P_neg", "P_neutral"]).copy()

# Parse topics column back to list (stored as string in csv)
import ast
bis_scored["topics"] = bis_scored["topics"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# For each topic, compute mean P_neg by country-year
agg_frames = []

for t in TOPICS:
    subset = bis_scored[bis_scored["topics"].apply(lambda x: t in x)].copy()
    agg = (subset.groupby(["iso", "year"])
                 .agg(
                     P_neg_mean   = ("P_neg",     "mean"),
                     P_pos_mean   = ("P_pos",     "mean"),
                     n_speeches   = ("P_neg",     "count")
                 )
                 .reset_index())
    agg.columns = ["iso", "year",
                   f"{t}_sent_neg", f"{t}_sent_pos", f"{t}_n_speeches"]
    agg_frames.append(agg)

# Merge all topics on iso-year
from functools import reduce
topic_sent = reduce(lambda l, r: pd.merge(l, r, on=["iso","year"], how="outer"),
                    agg_frames)

print(topic_sent.shape)
print(topic_sent.columns.tolist())
print(topic_sent.head(5))

(384, 17)
['iso', 'year', 'credit_sent_neg', 'credit_sent_pos', 'credit_n_speeches', 'real_estate_sent_neg', 'real_estate_sent_pos', 'real_estate_n_speeches', 'monetary_policy_sent_neg', 'monetary_policy_sent_pos', 'monetary_policy_n_speeches', 'sovereign_sent_neg', 'sovereign_sent_pos', 'sovereign_n_speeches', 'corporate_sent_neg', 'corporate_sent_pos', 'corporate_n_speeches']
   iso  year  credit_sent_neg  credit_sent_pos  credit_n_speeches  \
0  AUS  1998         0.249468         0.157739                1.0   
1  AUS  1999              NaN              NaN                NaN   
2  AUS  2000              NaN              NaN                NaN   
3  AUS  2001              NaN              NaN                NaN   
4  AUS  2002              NaN              NaN                NaN   

   real_estate_sent_neg  real_estate_sent_pos  real_estate_n_speeches  \
0                   NaN                   NaN                     NaN   
1                   NaN                   NaN             

In [4]:
# ── Load JST macro panel and merge ───────────────────────────────────────────
# Load the augmented macro panel from the dissertation pipeline
macro = pd.read_csv(os.path.join(DATA_DIR, "augmented_analysis", "df_macro_augmented.csv"))
print("Macro panel shape:", macro.shape)
print("Macro ISO codes:", sorted(macro["iso"].unique()))
print("Macro year range:", macro["year"].min(), "–", macro["year"].max())

Macro panel shape: (569, 101)
Macro ISO codes: ['AUS', 'BEL', 'CAN', 'CHE', 'DEU', 'DNK', 'ESP', 'FIN', 'FRA', 'GBR', 'IRL', 'ITA', 'JPN', 'NLD', 'NOR', 'PRT', 'SWE', 'USA']
Macro year range: 1989 – 2020


In [5]:
# ── Merge topic sentiment with macro panel ────────────────────────────────────
# Filter topic_sent to JST countries and 1989-2020 only
jst_countries = macro["iso"].unique().tolist()
topic_jst = topic_sent[
    (topic_sent["iso"].isin(jst_countries)) &
    (topic_sent["year"] >= 1989) &
    (topic_sent["year"] <= 2020)
].copy()

print("Topic sentiment (JST countries, 1989-2020):", topic_jst.shape)

# Merge
merged = pd.merge(macro, topic_jst, on=["iso", "year"], how="left")
print("Merged shape:", merged.shape)

# Coverage report
print("\nTopic sentiment coverage (% of macro rows with at least one score):")
for t in TOPICS:
    col = f"{t}_sent_neg"
    pct = 100 * merged[col].notna().sum() / len(merged)
    print(f"  {t:20s}: {pct:.1f}%")

Topic sentiment (JST countries, 1989-2020): (286, 17)
Merged shape: (569, 116)

Topic sentiment coverage (% of macro rows with at least one score):
  credit              : 19.3%
  real_estate         : 9.8%
  monetary_policy     : 35.0%
  sovereign           : 5.8%
  corporate           : 30.1%


In [6]:
# ── Coverage in Stage 2 window (2003-2020) ────────────────────────────────────
merged_s2 = merged[(merged["year"] >= 2003) & (merged["year"] <= 2020)]
print("Stage 2 rows:", len(merged_s2))

print("\nTopic sentiment coverage in Stage 2 window:")
for t in TOPICS:
    col = f"{t}_sent_neg"
    pct = 100 * merged_s2[col].notna().sum() / len(merged_s2)
    print(f"  {t:20s}: {pct:.1f}%")

print("\nCoverage by country (monetary_policy as proxy):")
print(merged_s2.groupby("iso")["monetary_policy_sent_neg"]
      .apply(lambda x: f"{100*x.notna().mean():.0f}%"))

Stage 2 rows: 319

Topic sentiment coverage in Stage 2 window:
  credit              : 30.1%
  real_estate         : 15.7%
  monetary_policy     : 53.3%
  sovereign           : 9.1%
  corporate           : 44.8%

Coverage by country (monetary_policy as proxy):
iso
AUS     17%
BEL     17%
CAN    100%
CHE     17%
DEU    100%
DNK     22%
ESP     40%
FIN     78%
FRA     44%
GBR    100%
IRL     50%
ITA     50%
JPN     62%
NLD      6%
NOR    100%
PRT     22%
SWE     33%
USA    100%
Name: monetary_policy_sent_neg, dtype: object


In [8]:
# ── Finalise: keep credit, monetary_policy, corporate only ────────────────────
keep_topics = ["credit", "monetary_policy", "corporate"]
drop_topics = ["real_estate", "sovereign"]

# Drop excluded topic columns
drop_cols = [c for c in merged.columns 
             if any(c.startswith(t) for t in drop_topics)]
merged_final = merged.drop(columns=drop_cols).copy()

# Add lag 1, 2, 3 versions of topic sentiment (consistent with dissertation)
for t in keep_topics:
    col = f"{t}_sent_neg"
    for lag in [1, 2, 3]:
        merged_final[f"{col}_lag{lag}"] = (merged_final
                                            .groupby("iso")[col]
                                            .shift(lag))

print("Final merged shape:", merged_final.shape)
print("\nNew topic sentiment columns:")
topic_cols = [c for c in merged_final.columns if "sent" in c]
print(topic_cols)

# Save
merged_final.to_csv(os.path.join(OUT_DIR, "jst_topic_sentiment_merged.csv"), 
                    index=False)
print("\nSaved: jst_topic_sentiment_merged.csv")

Final merged shape: (569, 119)

New topic sentiment columns:
['credit_sent_neg', 'credit_sent_pos', 'monetary_policy_sent_neg', 'monetary_policy_sent_pos', 'corporate_sent_neg', 'corporate_sent_pos', 'credit_sent_neg_lag1', 'credit_sent_neg_lag2', 'credit_sent_neg_lag3', 'monetary_policy_sent_neg_lag1', 'monetary_policy_sent_neg_lag2', 'monetary_policy_sent_neg_lag3', 'corporate_sent_neg_lag1', 'corporate_sent_neg_lag2', 'corporate_sent_neg_lag3']

Saved: jst_topic_sentiment_merged.csv
